# Contexta Research — 08 XLMR Supervised Training

Built on SenseForge shared modules. Run sequentially from the repository root or notebooks directory. Missing data is explicitly skipped; no fabricated results.

In [1]:
from pathlib import Path
import os
import sys
import json
import random
import gc

import numpy as np
import torch

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

sys.path.insert(0, str(ROOT / "src"))

ART = ROOT / "artifacts"
V2 = ART / "wsd_v2"
OUT = ART / "wsd_v2_supervised"

OUT.mkdir(parents=True, exist_ok=True)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

MODEL_NAME = "xlm-roberta-base"
MAX_LENGTH = 96

# CPU avoids previous Apple M2 MPS memory failures.
DEVICE = torch.device("cpu")

print("Project:", ROOT)
print("Dataset:", V2)
print("Output:", OUT)
print("Model:", MODEL_NAME)
print("Device:", DEVICE)

assert V2.exists(), "V2 dataset folder missing"

Project: /Users/suvratsmacbookair/MyLocalFiles/NLP project/SenseForge
Dataset: /Users/suvratsmacbookair/MyLocalFiles/NLP project/SenseForge/artifacts/wsd_v2
Output: /Users/suvratsmacbookair/MyLocalFiles/NLP project/SenseForge/artifacts/wsd_v2_supervised
Model: xlm-roberta-base
Device: cpu


In [3]:
from collections import Counter, defaultdict

def load_jsonl(path):
    with open(path, encoding="utf-8") as f:
        return [
            json.loads(line)
            for line in f
            if line.strip()
        ]

train_rows = load_jsonl(V2 / "train.jsonl")
val_rows = load_jsonl(V2 / "validation.jsonl")
test_rows = load_jsonl(V2 / "test.jsonl")

senses = sorted({
    row["sense_id"]
    for row in train_rows
})

words = sorted({
    row["target_word"]
    for row in train_rows
})

assert len(senses) == 12
assert len(words) == 6

for name, rows in [
    ("Train", train_rows),
    ("Validation", val_rows),
    ("Test", test_rows)
]:
    unknown = set(
        r["sense_id"] for r in rows
    ) - set(senses)

    assert not unknown, f"{name}: unseen labels {unknown}"

    print(f"\n{name}: {len(rows)} records")
    print("Classes:", len(set(r["sense_id"] for r in rows)))
    print("Sources:", dict(Counter(
        r["source_dataset"] for r in rows
    )))

print("\nWords:", words)
print("Senses:", senses)


Train: 506 records
Classes: 12
Sources: {'semcor_en': 264, 'wngt_examples_en': 23, 'wngt_glosses_en': 219}

Validation: 60 records
Classes: 12
Sources: {'xl_wsd_semcor_en': 34, 'semcor_en': 26}

Test: 62 records
Classes: 12
Sources: {'xl_wsd_semcor_en': 35, 'semcor_en': 27}

Words: ['bank', 'interest', 'plant', 'point', 'right', 'table']
Senses: ['bank.1', 'bank.2', 'interest.1', 'interest.2', 'plant.1', 'plant.2', 'point.1', 'point.2', 'right.1', 'right.2', 'table.1', 'table.2']


In [5]:
import hashlib

def normalize_text(text):
    return " ".join(text.casefold().split())

def text_hash(row):
    return hashlib.sha256(
        normalize_text(row["text_en"]).encode("utf-8")
    ).hexdigest()

splits = {
    "train": train_rows,
    "validation": val_rows,
    "test": test_rows
}

for name, rows in splits.items():
    for row in rows:
        start = row["target_char_start"]
        end = row["target_char_end"]

        assert 0 <= start < end <= len(row["text_en"])
        assert row["text_en"][start:end].strip()
        assert row["sense_id"].startswith(
            row["target_word"] + "."
        )

    print(f"{name}: target spans verified")

hashes = {
    name: {text_hash(r) for r in rows}
    for name, rows in splits.items()
}

for a, b in [
    ("train", "validation"),
    ("train", "test"),
    ("validation", "test")
]:
    overlap = hashes[a] & hashes[b]
    print(f"{a}/{b} exact overlap:", len(overlap))
    assert not overlap

print("\nBasic V2 integrity checks passed.")

train: target spans verified
validation: target spans verified
test: target spans verified
train/validation exact overlap: 0
train/test exact overlap: 0
validation/test exact overlap: 0

Basic V2 integrity checks passed.


In [7]:
from torch.utils.data import Dataset
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    use_fast=True
)

class WSDV2Dataset(Dataset):
    def __init__(self, rows, tokenizer, max_length=96):
        self.rows = rows
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, index):
        row = self.rows[index]
        text = row["text_en"]
        start = int(row["target_char_start"])
        end = int(row["target_char_end"])

        enc = self.tokenizer(
            text,
            add_special_tokens=True,
            truncation=False,
            return_offsets_mapping=True
        )

        ids = enc["input_ids"]
        offsets = enc["offset_mapping"]

        targets = [
            i for i, (a, b) in enumerate(offsets)
            if b > a and a < end and b > start
        ]

        if not targets:
            raise ValueError(
                f"Target alignment failed: {row['record_id']}"
            )

        # XLM-R uses start and end special tokens.
        capacity = self.max_length - 2
        first, last = targets[0], targets[-1]

        if last - first + 1 > capacity:
            raise ValueError(
                f"Target too long: {row['record_id']}"
            )

        content_end = len(ids) - 1
        window_start = max(1, first - capacity // 2)
        window_start = min(
            window_start,
            max(1, last - capacity + 1)
        )
        window_end = min(
            content_end,
            window_start + capacity
        )

        if window_end - window_start < capacity:
            window_start = max(
                1, window_end - capacity
            )

        positions = list(range(window_start, window_end))

        selected_ids = (
            [ids[0]]
            + [ids[i] for i in positions]
            + [ids[-1]]
        )

        target_set = set(targets)
        target_mask = (
            [0.0]
            + [float(i in target_set) for i in positions]
            + [0.0]
        )

        attention = [1] * len(selected_ids)

        pad = self.max_length - len(selected_ids)
        assert pad >= 0

        selected_ids += [tokenizer.pad_token_id] * pad
        attention += [0] * pad
        target_mask += [0.0] * pad

        assert len(selected_ids) == self.max_length
        assert len(attention) == self.max_length
        assert len(target_mask) == self.max_length
        assert sum(target_mask) > 0

        return {
            "input_ids": torch.tensor(
                selected_ids, dtype=torch.long
            ),
            "attention_mask": torch.tensor(
                attention, dtype=torch.long
            ),
            "target_mask": torch.tensor(
                target_mask, dtype=torch.float32
            )
        }

print("WSD V2 dataset adapter ready.")

WSD V2 dataset adapter ready.


In [9]:
datasets = {
    name: WSDV2Dataset(
        rows, tokenizer, max_length=MAX_LENGTH
    )
    for name, rows in splits.items()
}

for name, dataset in datasets.items():
    failures = []

    for i in range(len(dataset)):
        try:
            item = dataset[i]

            assert item["input_ids"].shape == (MAX_LENGTH,)
            assert item["attention_mask"].shape == (MAX_LENGTH,)
            assert item["target_mask"].shape == (MAX_LENGTH,)
            assert item["target_mask"].sum().item() > 0

        except Exception as e:
            failures.append((i, repr(e)))

    print(
        f"{name}: {len(dataset)} records, "
        f"{len(failures)} failures"
    )

    if failures:
        print("Examples:", failures[:5])

    assert not failures, f"{name} tokenization failed"

print("\nAll target-token alignments passed.")

train: 506 records, 0 failures
validation: 60 records, 0 failures
test: 62 records, 0 failures

All target-token alignments passed.


In [11]:
from transformers import AutoModel

gc.collect()

if torch.backends.mps.is_available():
    torch.mps.empty_cache()

encoder = AutoModel.from_pretrained(
    MODEL_NAME,
    attn_implementation="eager"
)

encoder.to(DEVICE)
encoder.eval()

for parameter in encoder.parameters():
    parameter.requires_grad = False

print("Encoder:", MODEL_NAME)
print("Hidden size:", encoder.config.hidden_size)
print("Device:", DEVICE)
print("Encoder frozen:", all(
    not p.requires_grad
    for p in encoder.parameters()
))

Encoder: xlm-roberta-base
Hidden size: 768
Device: cpu
Encoder frozen: True


In [13]:
from torch.utils.data import DataLoader

CACHE = OUT / "embedding_cache"
CACHE.mkdir(parents=True, exist_ok=True)

@torch.inference_mode()
def extract_embeddings(dataset, name):
    loader = DataLoader(
        dataset,
        batch_size=1,
        shuffle=False,
        num_workers=0
    )

    vectors = []

    encoder.eval()

    for i, batch in enumerate(loader):
        ids = batch["input_ids"].to(DEVICE)
        attention = batch["attention_mask"].to(DEVICE)
        target = batch["target_mask"].to(DEVICE)

        hidden = encoder(
            input_ids=ids,
            attention_mask=attention
        ).last_hidden_state

        weights = target.unsqueeze(-1)

        pooled = (
            (hidden * weights).sum(dim=1)
            / weights.sum(dim=1).clamp(min=1)
        )

        vectors.append(
            pooled.squeeze(0).cpu().numpy()
        )

        if (i + 1) % 25 == 0:
            print(
                f"{name}: {i + 1}/{len(dataset)}"
            )

    return np.stack(vectors).astype(np.float32)


def get_embeddings(name, dataset, rows):
    path = CACHE / f"{name}_xlmr.npy"
    manifest = CACHE / f"{name}_manifest.json"

    import hashlib

    fingerprint = hashlib.sha256(
        json.dumps(
            rows, sort_keys=True, ensure_ascii=False
        ).encode("utf-8")
    ).hexdigest()

    metadata = {
        "fingerprint": fingerprint,
        "model": MODEL_NAME,
        "max_length": MAX_LENGTH,
        "pooling": "target_mean",
        "dataset_adapter": "WSDV2Dataset_v1"
    }

    if path.exists() and manifest.exists():
        saved = json.loads(
            manifest.read_text(encoding="utf-8")
        )

        if saved == metadata:
            X = np.load(path)
            if X.shape == (
                len(rows), encoder.config.hidden_size
            ):
                print(f"Loaded cached {name}: {X.shape}")
                return X

    print(f"\nExtracting {name} embeddings...")
    X = extract_embeddings(dataset, name)

    np.save(path, X)
    manifest.write_text(
        json.dumps(metadata, indent=2),
        encoding="utf-8"
    )

    print(f"Saved {name}: {X.shape}")
    return X


# Extract training and validation only.
# Test embeddings remain untouched until final evaluation.

X_train = get_embeddings(
    "train", datasets["train"], train_rows
)

X_val = get_embeddings(
    "validation", datasets["validation"], val_rows
)

y_train = np.array([
    r["sense_id"] for r in train_rows
])

y_val = np.array([
    r["sense_id"] for r in val_rows
])

print("\nTraining embeddings:", X_train.shape)
print("Validation embeddings:", X_val.shape)


Extracting train embeddings...
train: 25/506
train: 50/506
train: 75/506
train: 100/506
train: 125/506
train: 150/506
train: 175/506
train: 200/506
train: 225/506
train: 250/506
train: 275/506
train: 300/506
train: 325/506
train: 350/506
train: 375/506
train: 400/506
train: 425/506
train: 450/506
train: 475/506
train: 500/506
Saved train: (506, 768)

Extracting validation embeddings...
validation: 25/60
validation: 50/60
Saved validation: (60, 768)

Training embeddings: (506, 768)
Validation embeddings: (60, 768)


In [15]:
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler, normalize
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)
import joblib

# Prototype baseline: training data only
prototype_labels = sorted(set(y_train))

prototype_vectors = []

for sense in prototype_labels:
    vectors = X_train[y_train == sense]
    centroid = vectors.mean(axis=0)
    prototype_vectors.append(centroid)

prototype_matrix = normalize(
    np.stack(prototype_vectors)
)

# Supervised models
models = {
    "logistic_regression": make_pipeline(
        StandardScaler(),
        LogisticRegression(
            C=0.1,
            class_weight="balanced",
            max_iter=3000,
            random_state=SEED
        )
    ),

    "linear_svm": make_pipeline(
        StandardScaler(),
        LinearSVC(
            C=0.01,
            class_weight="balanced",
            max_iter=10000,
            random_state=SEED
        )
    ),

    "mlp": make_pipeline(
        StandardScaler(),
        MLPClassifier(
            hidden_layer_sizes=(128,),
            activation="relu",
            alpha=1.0,
            learning_rate_init=0.001,
            early_stopping=False,
            max_iter=400,
            random_state=SEED
        )
    )
}

for name, model in models.items():
    print(f"\nTraining {name}...")
    model.fit(X_train, y_train)
    print("Completed:", name)

print("\nAll supervised models trained.")


Training logistic_regression...
Completed: logistic_regression

Training linear_svm...
Completed: linear_svm

Training mlp...
Completed: mlp

All supervised models trained.


In [17]:
def restricted_predictions(scores, classes, rows):
    classes = np.asarray(classes)
    predictions = []

    for score_row, row in zip(scores, rows):
        word = row["target_word"]

        allowed = np.array([
            str(c).startswith(word + ".")
            for c in classes
        ])

        if not allowed.any():
            raise ValueError(
                f"No trained senses for {word}"
            )

        masked = np.where(
            allowed, score_row, -np.inf
        )

        predictions.append(
            classes[np.argmax(masked)]
        )

    return np.asarray(predictions)


def predict_model(name, X, rows):
    if name == "prototypes":
        scores = (
            normalize(X) @ prototype_matrix.T
        )
        classes = prototype_labels

    else:
        model = models[name]
        classes = model.classes_

        if hasattr(model, "decision_function"):
            scores = model.decision_function(X)
        else:
            scores = model.predict_proba(X)

        scores = np.asarray(scores)

        if scores.ndim == 1:
            raise ValueError(
                "Expected multiclass score matrix"
            )

    return restricted_predictions(
        scores, classes, rows
    )


print("Lemma-restricted inference ready.")

Lemma-restricted inference ready.


In [19]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

results = []

for name in ["prototypes", *models.keys()]:
    pred = predict_model(name, X_val, val_rows)

    metrics = {
        "model": name,
        "accuracy": accuracy_score(y_val, pred),
        "macro_precision": precision_score(
            y_val, pred,
            labels=senses,
            average="macro",
            zero_division=0
        ),
        "macro_recall": recall_score(
            y_val, pred,
            labels=senses,
            average="macro",
            zero_division=0
        ),
        "macro_f1": f1_score(
            y_val, pred,
            labels=senses,
            average="macro",
            zero_division=0
        )
    }

    results.append(metrics)

results.sort(
    key=lambda x: (x["macro_f1"], x["accuracy"]),
    reverse=True
)

print("\n=== VALIDATION BENCHMARK ===")

for result in results:
    print(
        f"{result['model']:22} "
        f"Accuracy={result['accuracy']:.4f} "
        f"Precision={result['macro_precision']:.4f} "
        f"Recall={result['macro_recall']:.4f} "
        f"Macro-F1={result['macro_f1']:.4f}"
    )

best_name = results[0]["model"]

print("\nBest validation model:", best_name)

(OUT / "validation_results.json").write_text(
    json.dumps(results, indent=2),
    encoding="utf-8"
)


=== VALIDATION BENCHMARK ===
logistic_regression    Accuracy=1.0000 Precision=1.0000 Recall=1.0000 Macro-F1=1.0000
linear_svm             Accuracy=1.0000 Precision=1.0000 Recall=1.0000 Macro-F1=1.0000
mlp                    Accuracy=1.0000 Precision=1.0000 Recall=1.0000 Macro-F1=1.0000
prototypes             Accuracy=0.9000 Precision=0.8872 Recall=0.9147 Macro-F1=0.8911

Best validation model: logistic_regression


569

In [21]:
# FINAL TEST: Run once after model selection is frozen.

X_test = get_embeddings(
    "test", datasets["test"], test_rows
)

y_test = np.array([
    r["sense_id"] for r in test_rows
])

test_pred = predict_model(
    best_name, X_test, test_rows
)

test_metrics = {
    "model": best_name,
    "accuracy": float(
        accuracy_score(y_test, test_pred)
    ),
    "macro_precision": float(
        precision_score(
            y_test, test_pred,
            labels=senses,
            average="macro",
            zero_division=0
        )
    ),
    "macro_recall": float(
        recall_score(
            y_test, test_pred,
            labels=senses,
            average="macro",
            zero_division=0
        )
    ),
    "macro_f1": float(
        f1_score(
            y_test, test_pred,
            labels=senses,
            average="macro",
            zero_division=0
        )
    )
}

print("\n=== FINAL HELD-OUT TEST ===")

for key, value in test_metrics.items():
    print(key, value)

print("\nClassification report:")
print(classification_report(
    y_test, test_pred,
    labels=senses,
    zero_division=0
))

print("\nConfusion matrix:")
print(confusion_matrix(
    y_test, test_pred,
    labels=senses
))

(OUT / "test_metrics.json").write_text(
    json.dumps(test_metrics, indent=2),
    encoding="utf-8"
)

if best_name == "prototypes":
    np.savez_compressed(
        OUT / "best_prototypes.npz",
        labels=np.array(prototype_labels),
        vectors=prototype_matrix
    )
else:
    joblib.dump(
        models[best_name],
        OUT / "best_supervised_model.joblib"
    )

(OUT / "sense_labels.json").write_text(
    json.dumps(senses, indent=2),
    encoding="utf-8"
)

print("\nFinal artifacts saved:", OUT)


Extracting test embeddings...
test: 25/62
test: 50/62
Saved test: (62, 768)

=== FINAL HELD-OUT TEST ===
model logistic_regression
accuracy 1.0
macro_precision 1.0
macro_recall 1.0
macro_f1 1.0

Classification report:
              precision    recall  f1-score   support

      bank.1       1.00      1.00      1.00         3
      bank.2       1.00      1.00      1.00         3
  interest.1       1.00      1.00      1.00         3
  interest.2       1.00      1.00      1.00         4
     plant.1       1.00      1.00      1.00         3
     plant.2       1.00      1.00      1.00        14
     point.1       1.00      1.00      1.00        10
     point.2       1.00      1.00      1.00         7
     right.1       1.00      1.00      1.00         2
     right.2       1.00      1.00      1.00         1
     table.1       1.00      1.00      1.00         3
     table.2       1.00      1.00      1.00         9

    accuracy                           1.00        62
   macro avg       1.00

In [23]:
from collections import Counter, defaultdict
import numpy as np

print("=== PREDICTION AUDIT ===")

print("Selected model:", best_name)
print("Train records:", len(train_rows))
print("Validation records:", len(val_rows))
print("Test records:", len(test_rows))

print("\nTrue vs predicted labels:")
for word in sorted(set(r["target_word"] for r in test_rows)):
    indices = [
        i for i, row in enumerate(test_rows)
        if row["target_word"] == word
    ]

    truth = y_test[indices]
    predicted = test_pred[indices]

    correct = int(np.sum(truth == predicted))

    print(
        f"{word:12} "
        f"correct={correct}/{len(indices)} "
        f"true={dict(Counter(truth))} "
        f"pred={dict(Counter(predicted))}"
    )

print("\nFirst 10 predictions:")
for i in range(min(10, len(test_rows))):
    row = test_rows[i]
    print(
        row["target_word"],
        "| Gold:", y_test[i],
        "| Predicted:", test_pred[i],
        "| Text:", row["text_en"][:120]
    )

=== PREDICTION AUDIT ===
Selected model: logistic_regression
Train records: 506
Validation records: 60
Test records: 62

True vs predicted labels:
bank         correct=6/6 true={np.str_('bank.1'): 3, np.str_('bank.2'): 3} pred={np.str_('bank.1'): 3, np.str_('bank.2'): 3}
interest     correct=7/7 true={np.str_('interest.2'): 4, np.str_('interest.1'): 3} pred={np.str_('interest.2'): 4, np.str_('interest.1'): 3}
plant        correct=17/17 true={np.str_('plant.2'): 14, np.str_('plant.1'): 3} pred={np.str_('plant.2'): 14, np.str_('plant.1'): 3}
point        correct=17/17 true={np.str_('point.2'): 7, np.str_('point.1'): 10} pred={np.str_('point.2'): 7, np.str_('point.1'): 10}
right        correct=3/3 true={np.str_('right.1'): 2, np.str_('right.2'): 1} pred={np.str_('right.1'): 2, np.str_('right.2'): 1}
table        correct=12/12 true={np.str_('table.2'): 9, np.str_('table.1'): 3} pred={np.str_('table.2'): 9, np.str_('table.1'): 3}

First 10 predictions:
bank | Gold: bank.1 | Predicted: bank.